<a href="https://colab.research.google.com/github/olopez27CL/Lab01_EDA_Lopez_Omar/blob/main/mlp-mlflow-breast-cancer/IEI097_Lab2.12_MLP_MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2.12: Implementación de redes neuronales (MLP + MLflow)

**Nombre:** Omar Lopez  
**Número del seed:** 27092002  
**Fecha:** 09-11-2026  
**Acceso carpeta Github**: https://github.com/olopez27CL/Lab01_EDA_Lopez_Omar/tree/main/mlp-mlflow-breast-cancer

In [ ]:
!pip install -q mlflow pyngrok

In [ ]:
import random
import numpy as np
import pandas as pd
import sklearn
import mlflow

SEED = 27092002

random.seed(SEED)
np.random.seed(SEED)

print("Semilla:", SEED)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)

Semilla: 27092002
pandas: 2.2.3
scikit-learn: 1.6.1
MLflow: 3.18.0


In [ ]:
!pip install -q kagglehub

import os
import kagglehub

carpeta_datos = kagglehub.dataset_download(
    "uciml/breast-cancer-wisconsin-data"
)

print(os.listdir(carpeta_datos))

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
['data.csv']


In [ ]:
ruta_csv = os.path.join(carpeta_datos, "data.csv")
datos = pd.read_csv(ruta_csv)

print("Dimensiones:", datos.shape)
datos.info()

Dimensiones: (569, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  per

In [ ]:
print(datos.isna().sum())

datos = datos.drop(columns=["Unnamed: 32", "id"])

print("Dimensiones después de limpiar:", datos.shape)

id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst      0
Unnamed:

In [ ]:
datos["diagnosis"] = datos["diagnosis"].map({"M": 1, "B": 0})

print(datos["diagnosis"].value_counts())

diagnosis
0    357
1    212
Name: count, dtype: int64


### Balance de clases

El dataset contiene 357 casos benignos (clase 0) y 212 casos malignos (clase 1). Hay más casos benignos que malignos.

In [ ]:
variables = ["radius_mean", "area_mean", "smoothness_mean"]

resumen = datos[variables].agg(["min", "max"])
print(resumen)

rangos = resumen.loc["max"] - resumen.loc["min"]
print("\nRangos (máximo - mínimo):")
print(rangos)

print(
    "\nEl rango de area_mean es "
    f"{rangos['area_mean'] / rangos['smoothness_mean']:.0f} veces "
    "el rango de smoothness_mean."
)

     radius_mean  area_mean  smoothness_mean
min        6.981      143.5          0.05263
max       28.110     2501.0          0.16340

Rangos (máximo - mínimo):
radius_mean          21.12900
area_mean          2357.50000
smoothness_mean       0.11077
dtype: float64

El rango de area_mean es 21283 veces el rango de smoothness_mean.


### Diferencias de escala

`radius_mean` tiene un rango de 21.129, `area_mean` de 2357.5 y `smoothness_mean` de 0.11077. El rango de `area_mean` es aproximadamente 21 283 veces el de `smoothness_mean`, lo que evidencia que las variables están medidas en escalas muy diferentes.

In [ ]:
from sklearn.model_selection import train_test_split

X = datos.drop(columns=["diagnosis"])
y = datos["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

print("Tamaño de entrenamiento:", X_train.shape)
print("Tamaño de prueba:", X_test.shape)

print("\nProporciones en entrenamiento:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nProporciones en prueba:")
print(y_test.value_counts(normalize=True).sort_index())

Tamaño de entrenamiento: (455, 30)
Tamaño de prueba: (114, 30)

Proporciones en entrenamiento:
diagnosis
0    0.626374
1    0.373626
Name: proportion, dtype: float64

Proporciones en prueba:
diagnosis
0    0.631579
1    0.368421
Name: proportion, dtype: float64


In [ ]:
import os
import subprocess
import time
import requests
from google.colab import userdata
from pyngrok import ngrok

env = dict(
    os.environ,
    MLFLOW_SERVER_ALLOWED_HOSTS="*",
    MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*"
)

subprocess.Popen(
    [
        "mlflow", "server",
        "--backend-store-uri", "sqlite:///mlflow.db",
        "--host", "127.0.0.1",
        "--port", "5000"
    ],
    env=env
)

for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=1).ok:
            print("MLflow listo: 200")
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill()
tunel = ngrok.connect(5000)

print("MLflow UI:", tunel.public_url)

ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=5000): Read timed out. (read timeout=1)

In [ ]:
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill()
tunel = ngrok.connect(5000)

print("MLflow UI:", tunel.public_url)

MLflow UI: https://viable-unstable-deeply.ngrok-free.dev


In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
experimento = mlflow.set_experiment("IEI097-L2.12-Lopez")

print("Experimento activo:", experimento.name)

Experimento activo: IEI097-L2.12-Lopez


### Hipótesis para los runs A y B

Espero que el run B, con datos estandarizados, obtenga mejores resultados que el run A, sin estandarizar. Las variables tienen escalas muy distintas y el escalado podría facilitar el entrenamiento del MLP. Compararé las métricas para comprobar si se cumple esta hipótesis.

In [ ]:
import matplotlib.pyplot as plt
import mlflow

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def correr_experimento(
    nombre,
    escalar,
    arquitectura,
    activacion,
    alpha=0.0001,
    max_iter=500,
    random_state=SEED,
    semilla_particion=SEED,
    fase="experimentos"
):
    pasos = []

    if escalar:
        pasos.append(("escalador", StandardScaler()))

    pasos.append((
        "mlp",
        MLPClassifier(
            hidden_layer_sizes=arquitectura,
            activation=activacion,
            alpha=alpha,
            max_iter=max_iter,
            random_state=random_state
        )
    ))

    modelo = Pipeline(pasos)

    with mlflow.start_run(run_name=nombre):
        modelo.fit(X_train, y_train)

        pred_train = modelo.predict(X_train)
        pred_test = modelo.predict(X_test)
        mlp = modelo.named_steps["mlp"]

        mlflow.log_params({
            "arquitectura": str(arquitectura),
            "activacion": activacion,
            "alpha": alpha,
            "max_iter": max_iter,
            "random_state": random_state,
            "escalado": escalar,
            "semilla_particion": semilla_particion
        })

        accuracy_train = accuracy_score(y_train, pred_train)
        accuracy_test = accuracy_score(y_test, pred_test)
        recall_maligno = recall_score(y_test, pred_test, pos_label=1)
        f1 = f1_score(y_test, pred_test, pos_label=1)

        mlflow.log_metrics({
            "accuracy_train": accuracy_train,
            "accuracy_test": accuracy_test,
            "recall_maligno": recall_maligno,
            "f1": f1,
            "iteraciones": mlp.n_iter_
        })

        for epoca, perdida in enumerate(mlp.loss_curve_):
            mlflow.log_metric("train_loss", float(perdida), step=epoca)

        matriz = confusion_matrix(y_test, pred_test, labels=[0, 1])
        visualizacion = ConfusionMatrixDisplay(
            confusion_matrix=matriz,
            display_labels=["Benigno", "Maligno"]
        )

        visualizacion.plot(cmap="Blues", values_format="d")
        plt.title(f"Matriz de confusión - Run {nombre}")
        plt.savefig(f"matriz_confusion_{nombre}.png", bbox_inches="tight")
        mlflow.log_artifact(f"matriz_confusion_{nombre}.png")
        plt.close()

        mlflow.set_tag("fase", fase)

        print(
            f"Run {nombre} | "
            f"Accuracy prueba: {accuracy_test:.3f} | "
            f"Recall maligno: {recall_maligno:.3f} | "
            f"F1: {f1:.3f} | "
            f"Iteraciones: {mlp.n_iter_}"
        )

In [ ]:
correr_experimento(
    nombre="A",
    escalar=False,
    arquitectura=(30,),
    activacion="relu"
)

Run A | Accuracy prueba: 0.912 | Recall maligno: 0.857 | F1: 0.878 | Iteraciones: 217
🏃 View run A at: http://127.0.0.1:5000/#/experiments/1/runs/ba31da0bc7804d529db14e6a5923b1da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


El run A, sin estandarizar los datos, obtuvo una exactitud de prueba de 91.2%, un recall de la clase maligna de 85.7% y un F1 de 87.8%. Es el punto de partida para compararlo con el run B y comprobar la hipótesis sobre el escalado.

In [ ]:
correr_experimento(
    nombre="B",
    escalar=True,
    arquitectura=(30,),
    activacion="relu"
)

Run B | Accuracy prueba: 0.991 | Recall maligno: 1.000 | F1: 0.988 | Iteraciones: 368
🏃 View run B at: http://127.0.0.1:5000/#/experiments/1/runs/9c0d8b548f1d44b9852dd41eab4ae2aa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


El run B, con datos estandarizados, obtuvo mejores resultados que el run A: la exactitud de prueba subió de 91.2% a 99.1%, el recall maligno de 85.7% a 100% y el F1 de 87.8% a 98.8%. Esto coincide con la hipótesis de que el escalado ayudaría al MLP.

In [ ]:
correr_experimento(
    nombre="C",
    escalar=True,
    arquitectura=(64, 32),
    activacion="relu"
)

Run C | Accuracy prueba: 0.982 | Recall maligno: 1.000 | F1: 0.977 | Iteraciones: 197
🏃 View run C at: http://127.0.0.1:5000/#/experiments/1/runs/3d735ff120b44fd49a0e2a997591f482
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


El run C mantuvo un recall maligno de 100%, igual que el B, pero obtuvo menor exactitud de prueba (98.2% frente a 99.1%) y menor F1 (97.7% frente a 98.8%). En esta comparación, la arquitectura `(64, 32)` no mejoró el desempeño general.

In [ ]:
correr_experimento(
    nombre="D",
    escalar=True,
    arquitectura=(64, 32),
    activacion="tanh"
)

Run D | Accuracy prueba: 0.991 | Recall maligno: 1.000 | F1: 0.988 | Iteraciones: 333
🏃 View run D at: http://127.0.0.1:5000/#/experiments/1/runs/e13f1451c73c46a0b3698c39abe7e932
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


El run D obtuvo mejores resultados que el C: la exactitud de prueba subió de 98.2% a 99.1% y el F1 de 97.7% a 98.8%; el recall maligno se mantuvo en 100%. Sus métricas de prueba coinciden con las del run B.

In [ ]:
mlflow.end_run()

for semilla in range(SEED, SEED + 5):
    correr_experimento(
        nombre=f"C_semilla_{semilla}",
        escalar=True,
        arquitectura=(64, 32),
        activacion="relu",
        random_state=semilla,
        semilla_particion=SEED,
        fase="semillas"
    )

🏃 View run resilient-fly-995 at: http://127.0.0.1:5000/#/experiments/1/runs/a71356c716a34d5180d7dd01a2400ead
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run C_semilla_27092002 | Accuracy prueba: 0.982 | Recall maligno: 1.000 | F1: 0.977 | Iteraciones: 197
🏃 View run C_semilla_27092002 at: http://127.0.0.1:5000/#/experiments/1/runs/67a21f3a5a5046d8bc8516e9b5f8dcbd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run C_semilla_27092003 | Accuracy prueba: 0.991 | Recall maligno: 1.000 | F1: 0.988 | Iteraciones: 193
🏃 View run C_semilla_27092003 at: http://127.0.0.1:5000/#/experiments/1/runs/e2a637b987f440cfbe0585e99741855e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run C_semilla_27092004 | Accuracy prueba: 0.991 | Recall maligno: 1.000 | F1: 0.988 | Iteraciones: 197
🏃 View run C_semilla_27092004 at: http://127.0.0.1:5000/#/experiments/1/runs/473d12c61d6142458fdf2476021ce5f5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run C_semill

In [ ]:
correr_experimento(
    nombre=f"C_replica_{SEED}",
    escalar=True,
    arquitectura=(64, 32),
    activacion="relu",
    random_state=SEED,
    semilla_particion=SEED,
    fase="replica"
)

Run C_replica_27092002 | Accuracy prueba: 0.982 | Recall maligno: 1.000 | F1: 0.977 | Iteraciones: 197
🏃 View run C_replica_27092002 at: http://127.0.0.1:5000/#/experiments/1/runs/dde6148942694f90b871785115a38b22
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id]
)

metricas_comparar = [
    "metrics.accuracy_train",
    "metrics.accuracy_test",
    "metrics.recall_maligno",
    "metrics.f1",
    "metrics.iteraciones"
]

original = runs[
    runs["tags.mlflow.runName"] == "C"
].iloc[0]

replica = runs[
    runs["tags.mlflow.runName"] == f"C_replica_{SEED}"
].iloc[0]

print("Métricas de C original y su réplica:")
print(pd.DataFrame({
    "C original": original[metricas_comparar],
    "C réplica": replica[metricas_comparar]
}))

coinciden = (
    original[metricas_comparar] == replica[metricas_comparar]
).all()

print("\n¿Coinciden exactamente?:", coinciden)

semillas = runs[runs["tags.fase"] == "semillas"]

metricas_semillas = semillas[
    ["metrics.accuracy_test", "metrics.recall_maligno"]
]

print("\nCantidad de runs con semillas:", len(semillas))
print("\nMínimo y máximo:")
print(metricas_semillas.agg(["min", "max"]))

print("\nRango (máximo - mínimo):")
print(metricas_semillas.max() - metricas_semillas.min())

Métricas de C original y su réplica:
                       C original C réplica
metrics.accuracy_train        1.0       1.0
metrics.accuracy_test    0.982456  0.982456
metrics.recall_maligno        1.0       1.0
metrics.f1               0.976744  0.976744
metrics.iteraciones         197.0     197.0

¿Coinciden exactamente?: True

Cantidad de runs con semillas: 5

Mínimo y máximo:
     metrics.accuracy_test  metrics.recall_maligno
min               0.982456                     1.0
max               0.991228                     1.0

Rango (máximo - mínimo):
metrics.accuracy_test     0.008772
metrics.recall_maligno    0.000000
dtype: float64


### Variabilidad y replicación

En las cinco ejecuciones de C con distintas semillas, la exactitud de prueba varió entre 0.982456 y 0.991228, con un rango de 0.008772. El recall maligno fue 1.0 en todas las ejecuciones, con rango 0. La réplica de C con la semilla original coincidió exactamente con el run C en sus métricas, incluidas 0.982456 de exactitud de prueba y 197 iteraciones.

In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id]
)

columnas = [
    "run_id",
    "tags.mlflow.runName",
    "tags.fase",
    "params.arquitectura",
    "params.activacion",
    "params.alpha",
    "params.max_iter",
    "params.random_state",
    "params.escalado",
    "params.semilla_particion",
    "metrics.accuracy_train",
    "metrics.accuracy_test",
    "metrics.recall_maligno",
    "metrics.f1",
    "metrics.iteraciones",
    "metrics.train_loss"
]

runs_csv = runs[columnas]
runs_csv.to_csv("runs.csv", index=False)

print("Ejecuciones exportadas:", len(runs_csv))

from google.colab import files
files.download("runs.csv")

Ejecuciones exportadas: 11


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Preguntas de cierre

1. **¿Qué cambió al escalar las variables?**  
   El run A, sin escalado, obtuvo 91.2% de exactitud, 85.7% de recall maligno y 87.8% de F1. El run B, con escalado, obtuvo 99.1%, 100% y 98.8%, respectivamente. El *feature scaling* puso las variables, que tenían rangos muy distintos, en escalas comparables y coincidió con una mejora clara en estos resultados.

2. **¿Más capas y neuronas mejoraron el resultado?**  
   No se observa una mejora consistente. El run C, con arquitectura `(64, 32)`, obtuvo 98.2% de exactitud, algo menos que B, aunque mantuvo 100% de recall maligno. En las cinco semillas de C, la exactitud varió entre 98.2% y 99.1%; por eso, una sola ejecución no permite afirmar que la arquitectura más grande sea mejor.

3. **¿Cuánto variaron la exactitud y la sensibilidad entre las cinco semillas?**  
   La exactitud varió en un rango de 0.008772, equivalente a cerca de 0.88 puntos porcentuales. El recall maligno no varió: fue 1.0 en las cinco ejecuciones. La diferencia de exactitud entre B y C es similar a la variación observada entre semillas, mientras que la mejora de A a B fue mayor. Por eso, diferencias pequeñas entre configuraciones deben interpretarse considerando la variabilidad del modelo.

4. **¿Qué error es más costoso en el diagnóstico de cáncer? ¿Qué métrica priorizarías?**  
   Un falso negativo es más costoso: significa clasificar como benigno un tumor que realmente es maligno, lo que podría retrasar su atención. Por eso priorizaría el recall o sensibilidad de la clase maligna, que mide qué proporción de los casos malignos detecta el modelo.

5. **¿Qué información de MLflow no tendrías usando solo `print()`? ¿Qué más registrarías?**  
   MLflow conserva los parámetros y las métricas de cada ejecución, permite comparar runs, guardar la curva `train_loss` y consultar artefactos como la matriz de confusión. Con `print()` solo vería resultados en la salida de la celda, sin ese registro organizado. Para facilitar la replicación, también registraría la fuente y versión del dataset y las versiones de las librerías usadas.

In [28]:
ngrok.kill()
print("Túnel de ngrok cerrado.")

Túnel de ngrok cerrado.
